In [1]:
import requests

url = "https://api.sportradar.com/tennis/trial/v3/en/double_competitors_rankings.json"

headers = {
    "accept": "application/json",
    "x-api-key": "FvvDx4kWRqpl6Lg5z9u6tkCjEXYvo8ImNzEaFMQX"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    competitor_rankings_data = response.json()            # Step - 1 : Convert the response into dictionary format
    
    print(competitor_rankings_data)
else:
    print("Error in API:", response.status_code)


{'generated_at': '2026-04-07T06:39:44+00:00', 'rankings': [{'type_id': 2, 'name': 'ATP', 'year': 2026, 'week': 15, 'gender': 'men', 'competitor_rankings': [{'rank': 1, 'movement': 2, 'points': 8550, 'competitions_played': 25, 'competitor': {'id': 'sr:competitor:53785', 'name': 'Skupski, Neal', 'country': 'Great Britain', 'country_code': 'GBR', 'abbreviation': 'SKU'}}, {'rank': 2, 'movement': -1, 'points': 8430, 'competitions_played': 15, 'competitor': {'id': 'sr:competitor:16160', 'name': 'Zeballos, Horacio', 'country': 'Argentina', 'country_code': 'ARG', 'abbreviation': 'ZEB'}}, {'rank': 3, 'movement': -1, 'points': 8340, 'competitions_played': 15, 'competitor': {'id': 'sr:competitor:15568', 'name': 'Granollers, Marcel', 'country': 'Spain', 'country_code': 'ESP', 'abbreviation': 'GRA'}}, {'rank': 4, 'movement': 0, 'points': 7350, 'competitions_played': 22, 'competitor': {'id': 'sr:competitor:59131', 'name': 'Glasspool, Lloyd', 'country': 'Great Britain', 'country_code': 'GBR', 'abbrev

In [ ]:
competitor_rankings_data                     # Step - 2 : View the data and its structure.

In [3]:
# Step - 3 : PostgreSql connection and creating the table.

import psycopg2

connection = psycopg2.connect(
    host = "localhost",
    database = "sports",
    user = "postgres",
    password = "123456",
    port = "5432" )

cursor = connection.cursor()

cursor.execute('''CREATE TABLE IF NOT EXISTS competitors_table(
               competitor_id VARCHAR(50) PRIMARY KEY,
               name VARCHAR(100) NOT NULL,
               country VARCHAR(100) NOT NULL,
               country_code CHAR(3) NOT NULL,
               abbreviation VARCHAR(10) NOT NULL)''')


print("competitors table created successfully")


cursor.execute('''CREATE TABLE IF NOT EXISTS competitor_ranking_table (
               rank_id SERIAL PRIMARY KEY,rank INT NOT NULL,
               movement INT NOT NULL,points INT NOT NULL,
               competitions_played INT NOT NULL,competitor_id VARCHAR(50),
               FOREIGN KEY (competitor_id)
               REFERENCES competitors_table (competitor_id))''')


connection.commit()


print("competitior ranking table created successfully")

competitors table created successfully
competitior ranking table created successfully


In [4]:
#Step - 4 : Extract the competitors ranking from the given data.

for ranking in competitor_rankings_data['rankings']:

    for comp in ranking['competitor_rankings']:

        competitor = comp['competitor']

        competitor_id = competitor['id'].replace('sr:competitor:','')

        name = competitor['name']
        country = competitor['country']
        country_code = competitor.get('country_code','NO')
        abbreviation = competitor['abbreviation']

        rank = comp['rank']
        movement = comp['movement']
        points = comp['points']
        competitions_played = comp['competitions_played']

#Step - 5 : Competitors table inserting into the database.
        
        cursor.execute(''' INSERT INTO competitors_table
        (competitor_id,name,country,country_code,abbreviation)
        VALUES (%s,%s,%s,%s,%s) ON CONFLICT (competitor_id) DO NOTHING
        ''',(competitor_id,name,country,country_code,abbreviation))

#Step - 6 : Ranking table inserting into the database.
        
        cursor.execute(''' INSERT INTO competitor_ranking_table
        (rank,movement,points,competitions_played,competitor_id)
        VALUES (%s,%s,%s,%s,%s)''',(rank,movement,points,competitions_played,competitor_id))


connection.commit()


print("Competitors and ranking data inserted successfully")

Competitors and ranking data inserted successfully
